# BFSI Ticket Classifier — AMD vLLM
**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8
Only edit `VLLM_HOST` and `VLLM_MODEL` in Cell 1.

## Cell 1 — Config

In [ ]:
VLLM_HOST     = "http://localhost:8000"
VLLM_BASE_URL = f"{VLLM_HOST}/v1"
VLLM_MODEL    = "Qwen2.5-7B-Instruct"   # match --served-model-name exactly
VLLM_API_KEY  = "abc-123"

DB_PATH       = "complaints.duckdb"
JSON_PATH     = "synthetic_complaints.json"

CATEGORIES = [
    "fraud", "card_dispute", "kyc_onboarding", "loan_emi",
    "net_banking_upi", "branch_atm", "insurance", "general",
]
SLA_MAP  = {"P1": 1, "P2": 2, "P3": 5, "P4": 10}
TEAM_MAP = {
    "fraud": "Fraud Response Team", "card_dispute": "Card Operations",
    "kyc_onboarding": "KYC Team", "loan_emi": "Loan Servicing",
    "net_banking_upi": "Digital Banking", "branch_atm": "Branch Support",
    "insurance": "Insurance Claims", "general": "General Support",
}
CHANNELS  = ["web", "branch", "mobile_app"]
ACCOUNT_TYPES = ["savings", "current", "credit_card", "loan", "demat", "insurance"]
CATEGORY_WEIGHTS = {
    "fraud": 0.10, "card_dispute": 0.20, "net_banking_upi": 0.25,
    "kyc_onboarding": 0.10, "loan_emi": 0.15, "branch_atm": 0.08,
    "insurance": 0.07, "general": 0.05,
}
CATEGORY_ACCOUNT_AFFINITY = {
    "fraud": ["credit_card", "savings"], "card_dispute": ["credit_card", "savings"],
    "kyc_onboarding": ["savings", "current"], "loan_emi": ["loan"],
    "net_banking_upi": ["savings", "current"], "branch_atm": ["savings", "current"],
    "insurance": ["insurance"], "general": ["savings", "current", "credit_card"],
}
INDIAN_FIRST_NAMES = [
    "Rahul", "Priya", "Amit", "Sneha", "Vikram", "Kavya", "Rohan", "Pooja",
    "Arjun", "Deepika", "Suresh", "Anita", "Nikhil", "Meera", "Rajesh",
    "Sunita", "Kiran", "Neha", "Sanjay", "Divya", "Arun", "Lakshmi",
    "Manish", "Ritu", "Gaurav", "Swati", "Vinod", "Pallavi", "Ashok", "Rekha",
]
INDIAN_LAST_NAMES = [
    "Sharma", "Patel", "Singh", "Kumar", "Mehta", "Nair", "Iyer", "Joshi",
    "Reddy", "Verma", "Gupta", "Shah", "Pillai", "Rao", "Mishra", "Desai",
    "Malhotra", "Chopra", "Bose", "Das", "Pandey", "Sinha", "Tiwari", "Dubey",
]
EMAIL_DOMAINS  = ["gmail.com", "yahoo.com", "hotmail.com", "outlook.com", "rediffmail.com"]
MUMBAI_BRANCHES = [
    "Andheri West", "Bandra East", "Borivali", "Dadar", "Thane",
    "Kurla", "Powai", "Malad", "Goregaon", "Vashi", "Kandivali", "Mulund",
]
print("Config loaded.")

## Cell 2 — DB setup

In [ ]:
import duckdb

def get_conn():
    return duckdb.connect(DB_PATH)

def init_db():
    conn = get_conn()
    conn.execute("""
        CREATE TABLE IF NOT EXISTS complaints (
            complaint_id  VARCHAR PRIMARY KEY,
            customer_id   VARCHAR NOT NULL,
            cust_name     VARCHAR NOT NULL,
            cust_email    VARCHAR,
            complaint     TEXT    NOT NULL,
            channel       VARCHAR,
            branch        VARCHAR,
            account_type  VARCHAR,
            category      VARCHAR,
            priority      VARCHAR,
            team          VARCHAR,
            sentiment     VARCHAR,
            sla_days      INTEGER,
            summary       TEXT,
            reasoning     TEXT,
            status        VARCHAR DEFAULT 'open',
            created_at    TIMESTAMP DEFAULT current_timestamp,
            sla_deadline  TIMESTAMP
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS synthetic_raw (
            customer_id   VARCHAR,
            cust_name     VARCHAR,
            cust_email    VARCHAR,
            complaint     TEXT,
            category      VARCHAR,
            channel       VARCHAR,
            branch        VARCHAR,
            account_type  VARCHAR,
            generated_at  TIMESTAMP DEFAULT current_timestamp
        )
    """)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS classification_log (
            complaint_id  VARCHAR,
            raw_response  TEXT,
            parse_success BOOLEAN,
            latency_ms    INTEGER,
            model_used    VARCHAR,
            logged_at     TIMESTAMP DEFAULT current_timestamp
        )
    """)
    conn.close()
    print("DB initialised:", DB_PATH)

def insert_complaint(record):
    conn = get_conn()
    conn.execute("""
        INSERT OR REPLACE INTO complaints
            (complaint_id, customer_id, cust_name, cust_email, complaint,
             channel, branch, account_type, category, priority, team,
             sentiment, sla_days, summary, reasoning, status, created_at, sla_deadline)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                current_timestamp, current_timestamp + INTERVAL (?) DAY)
    """, [
        record["complaint_id"], record["customer_id"], record["cust_name"],
        record.get("cust_email"), record["complaint"],
        record.get("channel"), record.get("branch"), record.get("account_type"),
        record.get("category"), record.get("priority"), record.get("team"),
        record.get("sentiment"), record.get("sla_days"), record.get("summary"),
        record.get("reasoning"), "open", record.get("sla_days", 1)
    ])
    conn.close()

def log_classification(record):
    conn = get_conn()
    conn.execute("""
        INSERT INTO classification_log
            (complaint_id, raw_response, parse_success, latency_ms, model_used)
        VALUES (?, ?, ?, ?, ?)
    """, [
        record["complaint_id"], record["raw_response"],
        record["parse_success"], record["latency_ms"], record["model_used"]
    ])
    conn.close()

def load_json_to_db(json_path=JSON_PATH):
    """Load synthetic_complaints.json into synthetic_raw table."""
    import json, os
    if not os.path.exists(json_path):
        print(f"File not found: {json_path}")
        return
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    if not data:
        print("JSON file is empty.")
        return
    conn = get_conn()
    conn.executemany("""
        INSERT INTO synthetic_raw
            (customer_id, cust_name, cust_email, complaint, category, channel, branch, account_type)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, [
        (r["customer_id"], r["cust_name"], r.get("cust_email"), r["complaint"],
         r["category"], r["channel"], r.get("branch"), r["account_type"])
        for r in data
    ])
    conn.close()
    print(f"Loaded {len(data)} records from {json_path} into synthetic_raw.")

def fetch_queue(priority_filter=None, limit=50):
    conn = get_conn()
    if priority_filter:
        rows = conn.execute(
            "SELECT * FROM complaints WHERE priority=? ORDER BY created_at DESC LIMIT ?",
            [priority_filter, limit]
        ).fetchall()
    else:
        rows = conn.execute(
            "SELECT * FROM complaints ORDER BY created_at DESC LIMIT ?", [limit]
        ).fetchall()
    cols = [d[0] for d in conn.description]
    conn.close()
    return [dict(zip(cols, r)) for r in rows]

def fetch_stats():
    conn = get_conn()
    total    = conn.execute("SELECT COUNT(*) FROM complaints").fetchone()[0]
    by_cat   = dict(conn.execute("SELECT category, COUNT(*) FROM complaints GROUP BY category").fetchall())
    by_pri   = dict(conn.execute("SELECT priority, COUNT(*) FROM complaints GROUP BY priority").fetchall())
    breached = conn.execute(
        "SELECT COUNT(*) FROM complaints WHERE status='open' AND sla_deadline < current_timestamp"
    ).fetchone()[0]
    conn.close()
    return {"total": total, "by_category": by_cat, "by_priority": by_pri, "sla_breached": breached}

init_db()

## Cell 3 — Generator functions

In [ ]:
import random, json
from openai import OpenAI

client = OpenAI(base_url=VLLM_BASE_URL, api_key=VLLM_API_KEY)

CATEGORY_PROMPTS = {
    "fraud": """Generate 1 realistic Indian bank customer complaint about fraud or unauthorized transaction.
1-3 sentences, urgent tone. Occasional Hindi-English mix is fine.
Examples:
- My HDFC credit card was used for Rs 8500 at a Delhi merchant I never visited. Block immediately.
- Someone made 3 UPI transactions from my account totaling Rs 15000 last night.
Return ONLY the complaint text, nothing else.""",

    "card_dispute": """Generate 1 realistic Indian bank customer complaint about a card issue (blocked/declined/double charge).
1-3 sentences, frustrated tone.
Examples:
- My SBI debit card got blocked after paying at petrol pump. Need it unblocked urgently.
- Rs 2300 Zomato charge showing twice in my statement. I only ordered once.
Return ONLY the complaint text, nothing else.""",

    "kyc_onboarding": """Generate 1 realistic Indian bank customer complaint about KYC verification or account opening delay.
1-3 sentences, impatient tone.
Examples:
- I submitted Aadhaar and PAN for new savings account 3 weeks ago. Still showing under review.
- Video KYC appointment was scheduled but no one called. This is the third time rescheduling.
Return ONLY the complaint text, nothing else.""",

    "loan_emi": """Generate 1 realistic Indian bank customer complaint about loan EMI or home loan issue.
1-3 sentences, concerned tone.
Examples:
- Extra EMI of Rs 4200 deducted this month from home loan account. No prior notice given.
- I requested loan prepayment last month but full EMI still deducted.
Return ONLY the complaint text, nothing else.""",

    "net_banking_upi": """Generate 1 realistic Indian bank customer complaint about UPI failure or net banking issue.
1-3 sentences, frustrated tone.
Examples:
- Paid Rs 8500 to vendor via UPI. Money debited but vendor says not received.
- Net banking login stopped working after OTP verification. Getting error code 502.
Return ONLY the complaint text, nothing else.""",

    "branch_atm": """Generate 1 realistic Indian bank customer complaint about ATM or branch service.
1-3 sentences, annoyed tone.
Examples:
- ATM at Andheri West dispensed Rs 500 less but full amount got debited.
- Visited branch 3 times for fixed deposit issue. Every time told to come next day.
Return ONLY the complaint text, nothing else.""",

    "insurance": """Generate 1 realistic Indian bank customer complaint about insurance claim or policy issue.
1-3 sentences, distressed tone.
Examples:
- My father passed away in March. Life insurance claim submitted with all documents. Still pending 4 months.
- Health insurance cashless request rejected at hospital despite valid policy.
Return ONLY the complaint text, nothing else.""",

    "general": """Generate 1 realistic Indian bank customer complaint about a general service issue.
1-3 sentences, neutral or mildly frustrated tone.
Examples:
- Not receiving monthly account statement on registered email for last 2 months.
- Want to update mobile number linked to my account. Branch form process is too complicated.
Return ONLY the complaint text, nothing else.""",
}

def gen_customer_id():
    return str(random.randint(1000_0000_0000_0000, 9999_9999_9999_9999))

def gen_customer_name():
    return f"{random.choice(INDIAN_FIRST_NAMES)} {random.choice(INDIAN_LAST_NAMES)}"

def gen_email(name):
    if random.random() < 0.70:
        local = name.lower().replace(' ', '.') + str(random.randint(10, 99))
        return f"{local}@{random.choice(EMAIL_DOMAINS)}"
    return None

def gen_branch(channel):
    if channel == "branch":
        return random.choice(MUMBAI_BRANCHES)
    if random.random() < 0.20:
        return random.choice(MUMBAI_BRANCHES)
    return None

def generate_one(category):
    response = client.chat.completions.create(
        model=VLLM_MODEL,
        messages=[{"role": "user", "content": CATEGORY_PROMPTS[category]}],
        temperature=0.85,
        max_tokens=150,
    )
    text      = response.choices[0].message.content.strip().strip('"')
    channel   = random.choice(CHANNELS)
    cust_name = gen_customer_name()
    return {
        "customer_id":  gen_customer_id(),
        "cust_name":    cust_name,
        "cust_email":   gen_email(cust_name),
        "complaint":    text,
        "category":     category,
        "channel":      channel,
        "branch":       gen_branch(channel),
        "account_type": random.choice(CATEGORY_ACCOUNT_AFFINITY.get(category, ACCOUNT_TYPES)),
    }

def generate_dataset(count=80):
    categories  = list(CATEGORY_WEIGHTS.keys())
    weights     = list(CATEGORY_WEIGHTS.values())
    target_cats = random.choices(categories, weights=weights, k=count)
    dataset     = []
    for i, cat in enumerate(target_cats):
        try:
            record = generate_one(cat)
            dataset.append(record)
            print(f"[{i+1}/{count}] {cat} | {record['customer_id']} | {record['complaint'][:60]}...")
        except Exception as e:
            print(f"[{i+1}/{count}] FAILED ({cat}): {e}")
    return dataset

print("Generator ready.")

## Cell 4 — Generate → save JSON
Verify a few rows print before moving to Cell 5.

In [ ]:
COUNT = 80

dataset = generate_dataset(COUNT)

if not dataset:
    print("ERROR: dataset empty — check VLLM_MODEL name with: curl http://localhost:8000/v1/models")
else:
    with open(JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)
    print(f"\nSaved {len(dataset)} records to {JSON_PATH}")

## Cell 5 — Load JSON → DB
Reads the JSON file and inserts into `synthetic_raw`. Can re-run independently without regenerating.

In [ ]:
load_json_to_db(JSON_PATH)

# quick sanity check
conn  = get_conn()
count = conn.execute("SELECT COUNT(*) FROM synthetic_raw").fetchone()[0]
sample = conn.execute("SELECT customer_id, cust_name, category, complaint FROM synthetic_raw LIMIT 3").fetchall()
conn.close()

print(f"synthetic_raw row count: {count}")
for row in sample:
    print(row)

## Cell 6 — Classifier functions

In [ ]:
import time, uuid

CLASSIFY_SYSTEM_PROMPT = """You are a BFSI complaint classification and routing agent for an Indian bank.

Given a customer complaint, return ONLY a valid JSON object — no markdown, no explanation outside JSON.

JSON fields:
- category: one of [fraud, card_dispute, kyc_onboarding, loan_emi, net_banking_upi, branch_atm, insurance, general]
- priority: P1 (fraud/security, 1 day) | P2 (card block/KYC, 2 days) | P3 (UPI/EMI/ATM, 5 days) | P4 (general, 10 days)
- team: one of [Fraud Response Team, Card Operations, KYC Team, Loan Servicing, Digital Banking, Branch Support, Insurance Claims, General Support]
- sentiment: one of [angry, distressed, frustrated, neutral, satisfied]
- sla_days: 1 | 2 | 5 | 10
- summary: one sentence summarising the issue
- reasoning: 1-2 sentences explaining priority and team

Return ONLY valid JSON. No markdown fences. No preamble."""

def classify(complaint, customer_id, cust_name,
             cust_email=None, channel="web", branch=None, account_type="savings"):
    complaint_id = "CMP-" + str(uuid.uuid4())[:8].upper()
    user_msg     = f'Complaint: "{complaint}"\nChannel: {channel}\nAccount type: {account_type}'
    t0           = time.time()
    response     = client.chat.completions.create(
        model=VLLM_MODEL,
        messages=[
            {"role": "system", "content": CLASSIFY_SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0.1,
        max_tokens=512,
    )
    latency_ms = int((time.time() - t0) * 1000)
    raw        = response.choices[0].message.content.strip()

    try:
        result = json.loads(raw.replace("```json", "").replace("```", "").strip())
    except json.JSONDecodeError:
        log_classification({"complaint_id": complaint_id, "raw_response": raw,
                            "parse_success": False, "latency_ms": latency_ms,
                            "model_used": VLLM_MODEL})
        raise ValueError(f"Non-JSON response: {raw[:200]}")

    category = result.get("category", "general")
    if category not in CATEGORIES:
        category = "general"
    priority = result.get("priority", "P3")
    sla_days = SLA_MAP.get(priority, 5)
    team     = result.get("team", TEAM_MAP.get(category, "General Support"))

    record = {
        "complaint_id": complaint_id, "customer_id": customer_id,
        "cust_name": cust_name, "cust_email": cust_email,
        "complaint": complaint, "channel": channel, "branch": branch,
        "account_type": account_type, "category": category,
        "priority": priority, "team": team,
        "sentiment": result.get("sentiment", "neutral"),
        "sla_days": sla_days,
        "summary": result.get("summary", complaint[:100]),
        "reasoning": result.get("reasoning", ""),
    }
    insert_complaint(record)
    log_classification({"complaint_id": complaint_id, "raw_response": raw,
                        "parse_success": True, "latency_ms": latency_ms,
                        "model_used": VLLM_MODEL})
    record["latency_ms"] = latency_ms
    return record

def classify_from_json(json_path=JSON_PATH):
    """Read JSON file and classify each record into complaints table."""
    import os
    if not os.path.exists(json_path):
        print(f"File not found: {json_path}")
        return []
    with open(json_path, encoding="utf-8") as f:
        tickets = json.load(f)
    if not tickets:
        print("JSON file is empty.")
        return []
    results = []
    total   = len(tickets)
    for i, t in enumerate(tickets):
        try:
            r = classify(
                complaint=t["complaint"], customer_id=t["customer_id"],
                cust_name=t["cust_name"], cust_email=t.get("cust_email"),
                channel=t.get("channel", "web"), branch=t.get("branch"),
                account_type=t.get("account_type", "savings"),
            )
            results.append(r)
            print(f"[{i+1}/{total}] {r['complaint_id']} → {r['category']} {r['priority']} ({r['latency_ms']}ms)")
        except Exception as e:
            print(f"[{i+1}/{total}] FAILED: {e}")
            results.append({"error": str(e), "complaint": t.get("complaint", "")})
    return results

print("Classifier ready.")

## Cell 7 — Quick single test

In [ ]:
result = classify(
    complaint="My HDFC credit card was charged Rs 45000 at a Dubai merchant. I am currently in Mumbai.",
    customer_id="7823641509234871",
    cust_name="Rahul Sharma",
    cust_email="rahul.sharma42@gmail.com",
    channel="mobile_app",
    account_type="credit_card",
)
print(json.dumps(result, indent=2, default=str))

## Cell 8 — Classify full JSON file

In [ ]:
results = classify_from_json(JSON_PATH)
print(f"\nDone. {len(results)} processed.")

## Cell 9 — Inspect results

In [ ]:
import pandas as pd

print(json.dumps(fetch_stats(), indent=2))

df = pd.DataFrame(fetch_queue(limit=100))
df[["complaint_id", "cust_name", "category", "priority", "team", "sla_days", "status"]]